# Étape 00 — Transformation des données

Ce notebook fait, PAS À PAS, exactement ce que fait 'pipeline.py' d'un seul coup : transformer les
sources brutes ('data/donnees_brutes/') en tables prêtes à être utilisées par la suite du projet
('data/donnees_valides/'). Chaque section montre le résultat d'UNE fonction, avec une explication
avant le code qui l'appelle.

**Pour lancer tout le pipeline sans repasser par ce notebook** : 'python pipeline.py' (toute la
configuration - quels dossiers, quelles années - est réunie en haut de ce fichier).

**Note sur cet environnement de démonstration** : 'data/donnees_brutes/' n'est pas fourni avec ce
dépôt (sources tierces volumineuses ou soumises à condition d'usage - géométries IGN, comptages PNF,
extrait OpenStreetMap...). Ce notebook est donc écrit prêt à l'emploi, mais ses cellules ne
produiront un vrai résultat qu'une fois 'data/donnees_brutes/' rempli selon la disposition attendue
par chaque fonction (voir sa docstring).


## 0. Configuration

Les chemins et les années viennent de 'pipeline.py', pour ne JAMAIS avoir deux définitions
différentes du même chemin (une dans le notebook, une dans le pipeline) qui finiraient par diverger.


In [1]:
import sys
from pathlib import Path

ICI = Path.cwd()
for sous_dossier in ("d_zones", "d_debit", "d_geographie", "d_socio_eco"):
    sys.path.insert(0, str(ICI / sous_dossier))
sys.path.insert(0, str(ICI))

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

from pipeline import (
    DOSSIER_DONNEES_BRUTES, DOSSIER_DONNEES_VALIDES, DOSSIER_ZONES, DOSSIER_DEBIT_BRUT,
    DOSSIER_CAPTEURS, DOSSIER_GEOGRAPHIE, DOSSIER_SOCIO_ECO, ANNEES, MILLESIMES_POPULATION, FICHIER_OSM,
)

print("Données brutes  :", DOSSIER_DONNEES_BRUTES)
print("Données valides :", DOSSIER_DONNEES_VALIDES)
print("Années          :", ANNEES)


Données brutes  : C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_brutes
Données valides : C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides
Années          : (2019, 2020, 2021, 2022, 2023, 2024, 2025)


## 1. Zones administratives — ['d_zones/zones.py'](d_zones/zones.py)

'preparer_zones()' lit les géométries IGN (communes/départements/régions) et les tables de
population légale INSEE, calcule la surface de chaque zone en Lambert-93 (la projection officielle
française), puis retire les polygones qui en contiennent un autre - Paris, Lyon et Marseille
apparaissent à la fois comme UNE ville et comme leurs arrondissements ; on garde les arrondissements.
Enfin on ne garde que la France métropolitaine (Corse exclue).


In [2]:
from zones import preparer_zones

chemins_zones = preparer_zones(DOSSIER_DONNEES_BRUTES, DOSSIER_ZONES)
communes = pd.read_parquet(chemins_zones["communes"])
print(f"{len(communes):,} communes (métropole hors Corse)")
communes.drop(columns="geometry").head(5)


  [communes] 3 polygone(s) qui en contenaient un autre, retiré(s)


[zones] 34,428 communes -> C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\zones\communes.parquet
[zones] 94 départements -> C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\zones\departements.parquet


[zones] 12 régions -> C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\zones\regions.parquet


34,428 communes (métropole hors Corse)


,code_commune,nom_commune,code_departement,population_commune,aire_km2_commune
0,01001,L'Abergement-Clémenciat,01,875.0,15.619987
1,01002,L'Abergement-de-Varey,01,279.0,9.175445
2,01004,Ambérieu-en-Bugey,01,15930.0,24.509002
3,01005,Ambérieux-en-Dombes,01,1941.0,16.014150
4,01006,Ambléon,01,114.0,6.030705


## 2. Population — ['d_zones/population.py'](d_zones/population.py)

'preparer_population()' empile les classeurs INSEE de chaque millésime (2019 à 2023 - 2024 et 2025
réutilisent le dernier millésime publié, l'INSEE ayant un délai de publication) en une seule table
longue '(millésime, code_commune, population)'.


In [3]:
from population import preparer_population

chemin_population = preparer_population(DOSSIER_DONNEES_BRUTES, DOSSIER_ZONES, MILLESIMES_POPULATION)
population = pd.read_parquet(chemin_population)
print(f"{len(population):,} lignes, millésimes : {sorted(population['millesime'].unique())}")
population.head(3)


C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV5\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV5\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV5\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


[population] 5 millésimes, 174,808 lignes -> C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\zones\population.parquet
174,808 lignes, millésimes : [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


,millesime,code_commune,population
0,2019,01001,798.0
1,2019,01002,257.0
2,2019,01004,14514.0


## 3. Débits bruts des capteurs — ['d_debit/debits_bruts.py'](d_debit/debits_bruts.py)

'preparer_debits_bruts()' convertit le classeur Excel brut de chaque année (un export de la
plateforme nationale de comptage, ~5000 colonnes) en un format compact (un fichier par année) : les
débits horaires sont stockés en entiers 16 bits plutôt qu'en nombres à virgule, pour économiser la
place. Une année déjà convertie est sautée, pas reconvertie.


In [4]:
from debits_bruts import preparer_debits_bruts

chemins_debits = preparer_debits_bruts(DOSSIER_DONNEES_BRUTES, DOSSIER_DEBIT_BRUT, ANNEES)
print(f"{len(chemins_debits)} fichiers de débit annuel préparés")


[débits] 2019 déjà préparé
[débits] 2020 déjà préparé
[débits] 2021 déjà préparé
[débits] 2022 déjà préparé
[débits] 2023 déjà préparé
[débits] 2024 déjà préparé
[débits] 2025 déjà préparé
7 fichiers de débit annuel préparés


## 4. Capteurs — ['d_debit/capteurs.py'](d_debit/capteurs.py)

'preparer_capteurs()' fusionne les métadonnées des capteurs (nom, position, commune...) avec leur
débit horaire de chaque année (préparé à l'étape précédente) : un site peut avoir plusieurs "débits"
(plusieurs voies/sens de comptage), on les additionne heure par heure. Les sites qui n'ont jamais eu
le moindre débit sont retirés - ET les sites HORS FRANCE MÉTROPOLITAINE (Corse incluse) aussi, ICI, à
la source : c'est le point de départ de tout ce qui touche aux capteurs dans le reste du projet
(clustering, classification, BKT...), donc le plus tôt possible pour ce filtre.


In [5]:
from capteurs import preparer_capteurs

chemin_capteurs = preparer_capteurs(DOSSIER_DONNEES_BRUTES, DOSSIER_DEBIT_BRUT, DOSSIER_CAPTEURS, ANNEES)
print("écrit :", chemin_capteurs)


[capteurs] débits chargés pour 1891 sites distincts


C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\src\00_transformation_des_donnees\outils.py:30: RuntimeWarning: invalid value encountered in cast
  code = np.round(tableau * FLOW_SCALE).astype(np.uint16)


[capteurs] 1710 capteurs avec des données de débit -> C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\capteurs\sensors.msgpack (18 hors France métropolitaine écartés)


écrit : C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\capteurs\sensors.msgpack


## 5. Profils d'usage — ['d_debit/profils_usage.py'](d_debit/profils_usage.py)

'preparer_profils_usage()' résume le débit horaire de chaque (capteur, année) en profils à 3
échelles de temps (24 valeurs pour la journée, 168 pour la semaine, 52 pour l'année) plus 5 variables
spectrales - la matière première du clustering d'usage (étape suivante du projet, pas faite ici).


In [6]:
from profils_usage import preparer_profils_usage

chemin_profils = preparer_profils_usage(DOSSIER_CAPTEURS, DOSSIER_CAPTEURS, ANNEES)
profils = pd.read_parquet(chemin_profils)
print(f"{len(profils):,} profils (capteur, année)")
profils[["id_site", "annee", "d00", "d06", "d12", "d18", "s4"]].head(5)


[profils] 11,970 profils (249 colonnes chacun) -> C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\capteurs\usage_profiles.parquet
11,970 profils (capteur, année)


,id_site,annee,d00,d06,d12,d18,s4
0,200000071,2019,0.00000,0.000000,0.000000,0.000000,-0.000000
1,200000082,2019,0.00000,0.000000,0.000000,0.000000,-0.000000
2,200000083,2019,0.18806,0.179104,29.614925,26.922388,4.211233
3,200000084,2019,0.00000,0.000000,0.000000,0.000000,-0.000000
4,200000085,2019,0.00000,0.000000,0.000000,0.000000,-0.000000


## 6. Capteurs-années — [`d_debit/capteurs_annees.py`](d_debit/capteurs_annees.py)

`preparer_capteurs_annees()` calcule, pour chaque (capteur, année) : le débit annuel, la QTA (Quantité
de Trafic Annuel - déjà l'estimation annuelle correcte, voir la docstring du module), et si le capteur
est "actif" cette année-là (plus de 5% des heures avec du débit et plus de 100 passages au total) -
SEULS les capteurs actifs d'une année entrent dans le BKT observé de cette année-là (méthode retenue
par ce dépôt).

In [7]:
from capteurs_annees import preparer_capteurs_annees

chemin_capteurs_annees = preparer_capteurs_annees(DOSSIER_CAPTEURS, DOSSIER_CAPTEURS, ANNEES)
capteurs_annees = pd.read_parquet(chemin_capteurs_annees)
par_annee = capteurs_annees.groupby("annee").agg(capteurs_valides=("valid", "sum"), capteurs_actifs=("active", "sum"))
par_annee


[capteurs-années] 11,970 lignes (capteur, année), 7,852 actives


,capteurs_valides,capteurs_actifs
annee,,
2019,681,672
2020,914,795
2021,1092,1070
2022,1253,1227
2023,1408,1395
2024,1443,1402
2025,1319,1291


## 7. Géographie OSM : cours d'eau et mairies — ['d_geographie/geographie_osm.py'](d_geographie/geographie_osm.py)

'preparer_geographie_osm()' extrait d'un extrait OpenStreetMap (.osm.pbf) deux choses : les cours
d'eau/le littoral (pour la distance à l'eau, section suivante) et les mairies (pour la distance au
grand centre urbain, section 9). La lecture évite de charger tous les nœuds OSM en mémoire (un
demi-milliard) - on ne demande que les identifiants qui nous intéressent déjà (voir 'outils.py').


In [8]:
from geographie_osm import preparer_geographie_osm

chemin_eau_osm, chemin_mairies = preparer_geographie_osm(FICHIER_OSM, DOSSIER_ZONES, DOSSIER_GEOGRAPHIE)
print("cours d'eau/littoral :", chemin_eau_osm)
print("mairies              :", chemin_mairies)


cours d'eau/littoral : C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\geographie\waterways_coastline.gpkg
mairies              : C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\geographie\town_halls.parquet


## 8. Altitude — ['d_geographie/altitude.py'](d_geographie/altitude.py)

'preparer_altitude()' calcule le relief de chaque commune à partir du modèle SRTM (résolution 90 m) :
l'altitude de son centroïde, plus des statistiques sur toute sa surface (min/max/moyenne/écart-type
et l'étendue, le "dénivelé" - une commune est dite montagneuse si son étendue dépasse 200 m).


In [9]:
from altitude import preparer_altitude

chemin_altitude = preparer_altitude(DOSSIER_ZONES, DOSSIER_GEOGRAPHIE)
altitude = pd.read_parquet(chemin_altitude)
altitude.nlargest(3, "alt_range_m")[["code_commune", "alt_mean_m", "alt_range_m"]]


C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\src\00_transformation_des_donnees\d_geographie\altitude.py:98: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroides = communes.geometry.centroid


[altitude] 34,428 communes (MNT france_srtm.tif) -> C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\geographie\altitude.parquet


,code_commune,alt_mean_m,alt_range_m
28787,74236,1836.746230,4214.0
28626,74056,2454.194000,3788.0
28704,74143,1653.088156,3489.0


## 9. Distance à l'eau — ['d_geographie/eau.py'](d_geographie/eau.py)

'preparer_distances_eau()' mesure la distance du centroïde de chaque commune à la ligne (cours d'eau
préparés à la section 7) la plus proche de chaque type : côte, rivière, canal.


In [10]:
from eau import preparer_distances_eau

chemin_eau = preparer_distances_eau(DOSSIER_ZONES, DOSSIER_GEOGRAPHIE, DOSSIER_GEOGRAPHIE)
eau = pd.read_parquet(chemin_eau)
eau.nsmallest(3, "dist_coast_km")


C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\src\00_transformation_des_donnees\d_geographie\eau.py:28: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroides = gpd.GeoSeries(communes.geometry.centroid, crs="EPSG:4326").to_crs(2154)


[eau] distances calculées pour 34,428 communes -> C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\geographie\water_distance.parquet


,code_commune,dist_coast_km,dist_river_km,dist_canal_km
12451,33551,0.001,0.762,9.981
7539,22302,0.022,1.856,12.385
4403,13202,0.027,0.232,1.611


## 10. Socio-économie — ['d_socio_eco/socio_economique.py'](d_socio_eco/socio_economique.py)

'preparer_socio_economique()' joint la grille de densité INSEE (7 niveaux), le revenu médian, les
aménagements cyclables déclarés et les labels "accueil vélo" - tout indexé par code commune.


In [11]:
from socio_economique import preparer_socio_economique

chemin_socio = preparer_socio_economique(DOSSIER_DONNEES_BRUTES, DOSSIER_ZONES, DOSSIER_SOCIO_ECO)
socio = pd.read_parquet(chemin_socio)
socio[["code_commune", "LIBDENS7", "median_income"]].head(5)


C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\src\00_transformation_des_donnees\d_socio_eco\socio_economique.py:64: DtypeWarning: Columns (39: INSEE_EPCI) have mixed types. Specify dtype option on import or set low_memory=False.
  table = pd.read_csv(


[socio-éco] 34,428 communes x 74 colonnes -> C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\socio_economique\communes.parquet


,code_commune,LIBDENS7,median_income
0,01001,Rural à habitat dispersé,25820.0
1,01002,Rural à habitat dispersé,24480.0
2,01004,Centres urbains intermédiaires,21660.0
3,01005,Bourgs ruraux,24610.0
4,01006,Rural à habitat dispersé,24210.0


## 11. Distance urbaine (GCU) — ['d_geographie/urbain.py'](d_geographie/urbain.py)

'preparer_distance_urbaine()' mesure la distance de chaque commune à la mairie du "grand centre
urbain" (GCU) le plus proche - a besoin de la grille de densité (section précédente) jointe aux
communes, donc on la reconstruit ici comme le fait 'table_communes.py'.


In [12]:
import geopandas as gpd
from urbain import preparer_distance_urbaine

zones_geo = gpd.read_parquet(DOSSIER_ZONES / "communes.parquet")
avec_densite = zones_geo.merge(socio[["code_commune", "LIBDENS7"]], on="code_commune", how="left")
chemin_urbain = preparer_distance_urbaine(avec_densite, DOSSIER_GEOGRAPHIE, DOSSIER_GEOGRAPHIE)
pd.read_parquet(chemin_urbain).head(5)


C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\src\00_transformation_des_donnees\d_geographie\urbain.py:49: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroide = communes.geometry.centroid


[urbain] GCU : 733 de base + 304 promue(s)


[urbain] 296 groupes GCU


C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\src\00_transformation_des_donnees\d_geographie\urbain.py:49: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroide = communes.geometry.centroid


[urbain] distance au GCU pour 34,428 communes -> C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\geographie\urban_distance.parquet


,code_commune,dist_gcu_group_km_T15
0,01001,14.901257
1,01002,7.766412
2,01004,1.015253
3,01005,14.034736
4,01006,22.797736


## 12. Table finale des communes — ['table_communes.py'](table_communes.py)

'preparer_table_communes()' joint TOUT ce qui précède (zones, socio-économie, altitude, eau, urbain)
en une seule table, une ligne par commune, plus quelques variables dérivées (densité de population,
distances en échelle logarithmique, indicateurs côtier/montagneux) - c'est la table que les étapes
suivantes du projet (classification des communes, calcul du BKT) utiliseront comme entrée.


In [13]:
from table_communes import preparer_table_communes

chemin_table = preparer_table_communes(DOSSIER_DONNEES_BRUTES, DOSSIER_ZONES, DOSSIER_GEOGRAPHIE, DOSSIER_SOCIO_ECO, DOSSIER_DONNEES_VALIDES)
table_finale = pd.read_parquet(chemin_table)
print(f"table finale : {table_finale.shape[0]:,} communes x {table_finale.shape[1]} colonnes")
derivees = [c for c in table_finale.columns if c.startswith(("log_", "near_", "is_"))]
print("variables dérivées :", derivees)
table_finale.head(3)


C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\src\00_transformation_des_donnees\d_socio_eco\socio_economique.py:64: DtypeWarning: Columns (39: INSEE_EPCI) have mixed types. Specify dtype option on import or set low_memory=False.
  table = pd.read_csv(


[socio-éco] 34,428 communes x 74 colonnes -> C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\socio_economique\communes.parquet


C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\src\00_transformation_des_donnees\d_geographie\urbain.py:49: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroide = communes.geometry.centroid


[urbain] GCU : 733 de base + 304 promue(s)


[urbain] 296 groupes GCU


C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\src\00_transformation_des_donnees\d_geographie\urbain.py:49: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroide = communes.geometry.centroid


[urbain] distance au GCU pour 34,428 communes -> C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\geographie\urban_distance.parquet


[table communes] 34,428 communes x 100 colonnes -> C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\commune_features.parquet
table finale : 34,428 communes x 100 colonnes
variables dérivées : ['log_pop', 'log_pop_density', 'log_dist_coast', 'log_dist_river', 'log_dist_canal', 'near_coast_20km', 'is_mountainous']


,code_commune,nom_commune,code_departement,population_commune,aire_km2_commune,DENS,LIBDENS,DENS7,LIBDENS7,median_income,...,alt_range_m,alt_count_m,pop_density,log_pop,log_pop_density,log_dist_coast,log_dist_river,log_dist_canal,near_coast_20km,is_mountainous
0,01001,L'Abergement-Clémenciat,01,875.0,15.619987,3.0,Rural,6.0,Rural à habitat dispersé,25820.0,...,66.0,2834.0,56.017971,6.775366,4.043367,5.692681,1.270041,2.351185,0,0
1,01002,L'Abergement-de-Varey,01,279.0,9.175445,3.0,Rural,6.0,Rural à habitat dispersé,24480.0,...,456.0,1655.0,30.407245,5.634790,3.447039,5.664366,1.898819,2.029069,0,1
2,01004,Ambérieu-en-Bugey,01,15930.0,24.509002,2.0,Urbain intermédiaire,2.0,Centres urbains intermédiaires,21660.0,...,521.0,4310.0,649.965271,9.676022,6.478456,5.643371,0.957049,2.107300,0,1


## Et voilà

La table 'data/donnees_valides/commune_features.parquet' est prête. Pour tout relancer sans repasser
par ce notebook : 'python pipeline.py'. L'étape suivante du projet ('03_creation_du_jeu_de_donnees_cyclable/')
prépare le réseau cyclable - volontairement séparée de la transformation des données de base faite
ici.
